# Food segmentation v4 trên **Colab**: seed 1 và 2

1. Runtime → Change runtime type → **T4 GPU**
2. Tải `food_dataset_v4.zip` lên **thư mục gốc Google Drive** (My Drive)
3. Runtime → Run all. Kết quả và model được ghi thẳng vào `My Drive/food_v4_out/` sau mỗi seed, nên rớt kết nối cũng không mất.

Cấu hình giống hệt seed 0 đã chạy trên Kaggle: aug + cơm gộp, batch 16, 150 epoch, patience 50.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q ultralytics
import ultralytics, torch
print('ultralytics', ultralytics.__version__, '| cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
import glob, yaml, pathlib, shutil, os, collections, json
import numpy as np, pandas as pd, cv2

ZIP = '/content/drive/MyDrive/food_dataset_v4.zip'
assert os.path.exists(ZIP), 'Chua tai food_dataset_v4.zip len thu muc goc My Drive'
if not os.path.exists('/content/food_dataset_v4'):
    shutil.unpack_archive(ZIP, '/content')
OUTD = '/content/drive/MyDrive/food_v4_out'; os.makedirs(OUTD, exist_ok=True)
os.makedirs('/content/work', exist_ok=True)

SRC = '/content/food_dataset_v4'
cfg = yaml.safe_load(open(f'{SRC}/data.yaml'))
names = [cfg['names'][i] for i in sorted(cfg['names'])]
RICE = names.index('rice')

def write_yaml(root, path):
    c = dict(cfg); c['path'] = root
    yaml.safe_dump(c, open(path, 'w'), sort_keys=False, allow_unicode=True)
    return path

DATA = write_yaml(SRC, '/content/work/data_v4.yaml')
R1 = '/content/work/v4_rice1'
if not os.path.exists(R1):
    shutil.copytree(f'{SRC}/images', f'{R1}/images')
    shutil.copytree(f'{SRC}/labels_rice1', f'{R1}/labels')
DATA_R1 = write_yaml(R1, '/content/work/data_rice1.yaml')
for s in ('train', 'val', 'test'):
    print(s, len(glob.glob(f'{SRC}/images/{s}/*')), 'anh')

## Tham số chung + cách chấm điểm

In [ ]:
from ultralytics import YOLO

COMMON = dict(epochs=150, patience=50, imgsz=640, batch=16, device=0, workers=2,
              deterministic=True, project='/content/work/runs', exist_ok=True, plots=False)
AUG = dict(degrees=180, flipud=0.5, fliplr=0.5, scale=0.6, copy_paste=0.3, cos_lr=True, close_mosaic=25)
CONFIGS = {'final': (DATA_R1, AUG)}
SEEDS = [1, 2]
CONF = 0.25   # chốt trước, dùng chung cho mọi cấu hình

TEST_IMGS = sorted(glob.glob(f'{SRC}/images/test/*.jpg'))

def gt_masks(img_path, shape):
    # Đáp án gốc (cơm nhiều mảnh) -> mask hợp theo lớp. Dùng chung cho MỌI cấu hình.
    H, W = shape
    out = {}
    lp = img_path.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
    for l in open(lp):
        r = l.split()
        if not r: continue
        c = int(r[0]); xy = (np.array(r[1:], float).reshape(-1, 2) * [W, H]).round().astype(np.int32)
        m = out.setdefault(c, np.zeros((H, W), np.uint8)); cv2.fillPoly(m, [xy], 1)
    return out

def free_metrics(w):
    model = YOLO(w)
    inter, union = collections.Counter(), collections.Counter()
    rice_iou, rice_err = [], []
    for p in TEST_IMGS:
        r = model.predict(p, conf=CONF, retina_masks=True, verbose=False)[0]
        H, W = r.orig_shape
        pred = {}
        if r.masks is not None:
            for m, c in zip(r.masks.data.cpu().numpy() > 0.5, r.boxes.cls.cpu().numpy().astype(int)):
                pred[c] = pred.get(c, np.zeros((H, W), bool)) | m
        gt = gt_masks(p, (H, W))
        for c in set(gt) | set(pred):
            g = gt.get(c, np.zeros((H, W), np.uint8)).astype(bool); q = pred.get(c, np.zeros((H, W), bool))
            inter[c] += (g & q).sum(); union[c] += (g | q).sum()
        if RICE in gt:
            g = gt[RICE].astype(bool); q = pred.get(RICE, np.zeros((H, W), bool))
            rice_iou.append((g & q).sum() / max((g | q).sum(), 1))
            rice_err.append(abs(int(q.sum()) - int(g.sum())) / g.sum())
    miou = np.mean([inter[c] / union[c] for c in union if union[c]])
    return dict(rice_iou=np.mean(rice_iou), rice_area_err=np.mean(rice_err), mIoU_all=miou)

def evaluate(w, data, tag):
    out = {}
    for split in ('val', 'test'):
        v = YOLO(w).val(data=data, split=split, imgsz=640, plots=False, verbose=False)
        out[f'{split}_mask_mAP50_95'] = v.seg.map; out[f'{split}_box_mAP50_95'] = v.box.map
        if split == 'test':
            ap = dict(zip(v.ap_class_index, v.seg.ap))
            out['test_mask_AP_excl_rice'] = np.mean([a for c, a in ap.items() if c != RICE])
            per = pd.DataFrame({'lop': [names[c] for c in ap], tag: list(ap.values())})
    out.update(free_metrics(w))
    return out, per

## Train model cuối

In [ ]:
rows, per_all = [], None
if os.path.exists(f'{OUTD}/summary.csv'):
    rows = pd.read_csv(f'{OUTD}/summary.csv').to_dict('records')
done = {(r['cfg'], r['seed']) for r in rows}

for sd in SEEDS:
    for name, (data, aug) in CONFIGS.items():
        if (name, sd) in done: continue
        tag = f'{name}_s{sd}'
        w = f'/content/work/runs/{tag}/weights/best.pt'
        try:
            YOLO('yolo26s-seg.pt').train(data=data, name=tag, seed=sd, **COMMON, **aug)
            # lưu model TRƯỚC khi chấm: chấm lỗi thì chấm lại được, không phải train lại
            shutil.copy2(w, f'{OUTD}/{tag}_best.pt')
            shutil.copy2(f'/content/work/runs/{tag}/results.csv', f'{OUTD}/{tag}_results.csv')
        except Exception as e:
            print('LOI TRAIN', tag, repr(e)); rows.append(dict(cfg=name, seed=sd, error='train: ' + repr(e)[:200]))
            pd.DataFrame(rows).to_csv(f'{OUTD}/summary.csv', index=False); continue
        try:
            m, per = evaluate(w, data, tag)
            rows.append(dict(cfg=name, seed=sd, **m))
            per_all = per if per_all is None else per_all.merge(per, on='lop', how='outer')
        except Exception as e:
            print('LOI CHAM', tag, repr(e)); rows.append(dict(cfg=name, seed=sd, error='eval: ' + repr(e)[:200]))
        pd.DataFrame(rows).to_csv(f'{OUTD}/summary.csv', index=False)
        if per_all is not None: per_all.to_csv(f'{OUTD}/per_class_test_ap.csv', index=False)
        print(pd.DataFrame(rows).round(3).to_string())

## Kết quả

In [ ]:
S = pd.DataFrame(rows)
cols = [c for c in S.columns if c not in ('cfg', 'seed', 'error')]
S.round(3).T